# Do dermatology VLMs look at the lesion?

Pilot run. Needs a GPU (T4 is enough), internet on, the Hugging Face token saved as the Kaggle secret `HF_TOKEN`
(with the MedGemma terms accepted on Hugging Face), and two private Kaggle datasets: Derm7pt and PAD-UFES-20.

Run once with `MODEL = "qwen"` and once with `MODEL = "medgemma"`. Each run takes a few hours; use **Save & Run All**
so it keeps going in the background. Results are written to `/kaggle/working/` and the run can be restarted:
answers already written are skipped.

In [ ]:
MODEL = "qwen"        # "qwen" or "medgemma"
LIMIT = None          # set to e.g. 5 for a quick test of the whole pipeline

In [ ]:
!git clone -q https://github.com/subarna007/derm-vlm-shortcut-pilot /kaggle/working/pilot 2>/dev/null || git -C /kaggle/working/pilot pull -q
%cd /kaggle/working/pilot
!pip install -q -U "transformers>=4.53" accelerate bitsandbytes

In [ ]:
import os, glob
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

# find the two datasets wherever they were attached
d7 = [p for p in glob.glob("/kaggle/input/**/meta.csv", recursive=True) if "/meta/" in p]
pad = glob.glob("/kaggle/input/**/metadata.csv", recursive=True)
assert d7 or pad, "attach the PAD-UFES-20 and/or Derm7pt datasets to this notebook"
os.environ["DERM7PT_DIR"] = os.path.dirname(os.path.dirname(d7[0])) if d7 else "/missing"
os.environ["PAD_DIR"] = os.path.dirname(pad[0]) if pad else "/missing"
os.environ["WORK_DIR"] = "/kaggle/working/work"
os.environ["RESULTS_DIR"] = "/kaggle/working/pilot/results"
# Derm7pt can also be downloaded directly, with the secrets DERM7PT_USER and DERM7PT_PASS
if not d7:
    from src.fetch import setup_derm7pt
    setup_derm7pt()
print(os.environ["DERM7PT_DIR"], os.environ["PAD_DIR"])

Sample the images, make lesion masks and the four edited versions of each image. Skipped if already done.

In [ ]:
import importlib, pathlib
from src import config, run
importlib.reload(config)
if not pathlib.Path("/kaggle/working/work/manifest.csv").exists():
    run.prepare(config.DERM7PT_DIR, config.PAD_DIR, config.WORK_DIR)

Check the masks. Write the ids of any bad ones, one per line, into `results/bad_masks.txt` and rerun the cell above after deleting `work/manifest.csv`.

In [ ]:
from IPython.display import Image, display
for p in sorted(glob.glob("/kaggle/working/work/mask_qc_*.jpg"))[:2]:
    display(Image(p, width=900))

In [ ]:
from src.models import VLM
model = VLM(MODEL)
print(model.key, "4-bit" if model.four_bit else "", model.dtype)
run.infer(model, work=config.WORK_DIR, limit=LIMIT)

In [ ]:
from src import analyze
analyze.main(work=config.WORK_DIR, results=config.RESULTS_DIR)
!cd /kaggle/working && zip -qr pilot_outputs.zip work/raw work/manifest.csv pilot/results